# Evaluate Climate Models

### Parameter Settings: Change Timestep here

In [ ]:
# %% [Setup — Climate Model Evaluation]

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

FIG_SUBDIR = "climate_model_evaluation"
FIG_DIRS = [
    cfg.FIGURES_DIR           / FIG_SUBDIR,
    cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR,
]
for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# %% [Load per-catchment data for evaluation figures]
#
# Loads annual maxima and daily values per catchment, per model, for the
# evaluation period. Results are used by:
#   - make_distribution_figure  (distribution analysis plots)
#   - make_qq_figure            (Q-Q mapping plots)
#   - build_percentile_mapping_table / build_distribution_summary_table

from catchment_tools import (
    load_annual_maxima_per_catchment,
    load_daily_values_per_catchment,
    build_percentile_mapping_table,
    build_distribution_summary_table,
)
from plot_style import make_distribution_figure, make_qq_figure

# ── Evaluation period ─────────────────────────────────────────────────────────
# Must be contained within both the SMILE and reanalysis cached ranges.
# Adjust if your cached data uses a different overlap period.
EVAL_START_YEAR = 1985
EVAL_END_YEAR   = 2024
EVAL_PERIOD_TAG = f"{EVAL_START_YEAR}-{EVAL_END_YEAR}"

REANALYSIS_KEYS     = ["senorge", "era5_0.25", "era5_0.5"]
PERCENTILES_TO_COMPARE = (2.5, 5, 16, 50, 84, 95, 97.5)

# ── Load annual maxima per catchment ─────────────────────────────────────────
print("Loading 1-day annual maxima per catchment ...")
am_1day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day annual maxima per catchment ...")
am_2day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

# ── Load daily values per catchment ──────────────────────────────────────────
print("Loading 1-day daily values per catchment ...")
daily_1day_per_catchment = load_daily_values_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day daily values per catchment ...")
daily_2day_per_catchment = load_daily_values_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print(f"\n✓ Done. Eval period: {EVAL_PERIOD_TAG}")
print(f"  Catchments : {list(cfg.CATCHMENTS.keys())}")
for slug in cfg.CATCHMENTS:
    models_1day = list(am_1day_per_catchment[slug].keys())
    print(f"  {slug}: {models_1day}")

### Create Distribution plots

In [ ]:
# %% [Distribution Analysis — per catchment, per window, annual_max + daily]
#
# Produces one PDF per (catchment × window_days × data_type) combination.
#
# Filename pattern:
#   {data_type}_distribution_{window_days}day_{slug}_{start}-{end}.pdf
#
# Saved to: FIG_DIRS (climate_model_evaluation/, NO subfolder)

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),
    ]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),
        ]:
            data_for_catchment = pc[slug]

            if not data_for_catchment:
                print(f"  [skip] No data for {slug} / {window_days}day / {data_type}")
                continue

            fname = (
                f"{data_type}_distribution_{window_days}day_"
                f"{slug}_{EVAL_PERIOD_TAG}.pdf"
            )
            out_paths = [d / fname for d in FIG_DIRS]

            print(f"  Saving: {fname}")
            make_distribution_figure(
                annual_maxima   = data_for_catchment,
                window_days     = window_days,
                out_paths       = out_paths,
                data_type       = data_type,
                catchment_title = catchment_title,
            )

print(f"\n[distribution] ✓ Done. Figures saved to:")
for d in FIG_DIRS:
    print(f"  {d}")

### Create Q-Q mapping Plots

In [ ]:
# Per-catchment QQ plots: CESM2-LE and GFDL-SPEAR vs. Reanalysis
# Runs for both annual_max and daily data types.

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),]:
            for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
                reanalysis = {k: pc[slug][k] for k in REANALYSIS_KEYS if k in pc[slug]}
                out = [
                    d / f"qq-plot_{data_type}_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.pdf"
                    for d in FIG_DIRS]
                make_qq_figure(
                    climate_key,
                    pc[slug][climate_key],
                    reanalysis,
                    window_days=window_days,
                    out_paths=out,
                    data_type=data_type,
                    catchment_title=catchment_title,)

### Create Statistical Analysis for QQ-Plot

In [ ]:
# Per-catchment percentile mapping tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        refs = {k: am_pc[slug][k] for k in REANALYSIS_KEYS if k in am_pc[slug]}

        for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
            df_pct = build_percentile_mapping_table(
                climate_key,
                am_pc[slug][climate_key],
                refs,
                percentiles=PERCENTILES_TO_COMPARE,
            ).round(1)

            print(f"\nPercentile mapping: {climate_key} | {window_days}-day | {slug} | {EVAL_PERIOD_TAG}")
            display(df_pct)

            for d in FIG_DIRS:
                out = d / f"percentile_mapping_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
                df_pct.to_csv(out, index=False)
                print(f"Saved -> {out}")

In [ ]:
# Per-catchment distribution summary tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        summary = build_distribution_summary_table(am_pc[slug]).round(2)

        print(f"\n{window_days}-day summary | {slug} | {EVAL_PERIOD_TAG}")
        display(summary)

        for d in FIG_DIRS:
            out = d / f"distribution_summary_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
            summary.to_csv(out, index=False)
            print(f"Saved -> {out}")